# Descriptive analysis

Describes the participants (Table 1) and the answers to every question, overall and by group, with 95% confidence intervals (Wilson method). It also describes how participants combined their answers on accountability (Q11). No statistical tests are run here.

**Reads:** `data/survey_clean.csv`

**Writes to `results/`:** `table1_characteristics.csv`, `item_distributions.csv`, `key_percentages.csv`, `ordered_item_summaries.csv`, `accountability_patterns.csv`

In [1]:
from pathlib import Path

import pandas as pd
from statsmodels.stats.proportion import proportion_confint

DATA_FILE = Path("../data/survey_clean.csv")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

GROUPS = ["Patient", "Healthcare professional", "AI developer"]

# Answer options in the order used in the analysis.
# For Q7, Q9, Q14 and Q16 this is meaning order (Not < Neutral < Somewhat < ... < Extremely);
CATEGORIES = {
    "gender": ["Male", "Female", "Non-binary", "Prefer not to say"],
    "age_group": ["18-24", "25-34", "35-44", "45-54", "55-64", "65 and above"],
    "experience": ["Less than 1 year", "1-3 years", "4-6 years", "7-10 years", "More than 10 years"],
    "familiarity": ["Not familiar at all", "Somewhat familiar", "Moderately familiar", "Familiar", "Very familiar"],
    "bias_concern": ["Not concerned", "Neutral", "Somewhat concerned", "Concerned", "Extremely concerned"],
    "privacy_protection": ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"],
    "explanation_importance": ["Not important", "Neutral", "Somewhat important", "Important", "Extremely important"],
    "inform_patients": ["Yes", "No", "Not sure"],
    "trust": ["No trust", "Little trust", "Complete trust"],
    "autonomy_undermined": ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"],
    "guidelines_confidence": ["Not confident", "Neutral", "Somewhat confident", "Confident", "Extremely confident"],
    "no_human_life_death": ["Yes", "No", "Not sure"],
    "disparities_concern": ["Not concerned", "Neutral", "Somewhat concerned", "Concerned", "Extremely concerned"],
    "continent": ["Africa", "Asia", "Europe", "North America", "South America", "Oceania (Australia and the Pacific)"],
}

## Load the data

In [2]:
df = pd.read_csv(DATA_FILE)
for var, cats in CATEGORIES.items():
    assert set(df[var].dropna()) <= set(cats), var
    df[var] = pd.Categorical(df[var], categories=cats, ordered=True)
df["group"] = pd.Categorical(df["group"], categories=GROUPS)
print("Participants:", len(df))
df["group"].value_counts(sort=False)

Participants: 253


group
Patient                     48
Healthcare professional     58
AI developer               147
Name: count, dtype: int64

## Helper functions

In [3]:
def n_pct(n, d):
    """Return "n (%)" text."""
    return str(int(n)) + " (" + format(100 * n / d, ".1f") + ")" if d else "0 (-)"


def wilson(k, n):
    """95% confidence interval for a percentage (Wilson method)."""
    lo, hi = proportion_confint(int(k), int(n), alpha=0.05, method="wilson")
    return 100 * lo, 100 * hi


def by_group(frame):
    """Yield (label, subset) for each group and then for all participants."""
    for g in GROUPS:
        yield g, frame[frame["group"] == g]
    yield "Total", frame

## Table 1: Characteristics of participants

In [4]:
EXPERIENCE_ROWS = CATEGORIES["experience"] + ["Not answered"]
rows = [{"characteristic": "Participants, n", "category": "", **{g: str(len(sub)) for g, sub in by_group(df)}}]

for var, label in [("gender", "Gender"), ("age_group", "Age group (years)"),
                   ("continent", "Continent of residence"), ("familiarity", "Familiarity with AI in healthcare")]:
    rows.append({"characteristic": label, "category": "", **{g: "" for g in GROUPS + ["Total"]}})
    for cat in CATEGORIES[var]:
        rows.append({"characteristic": "", "category": cat,
                     **{g: n_pct((sub[var] == cat).sum(), len(sub)) for g, sub in by_group(df)}})

rows.append({"characteristic": "Experience (professionals and developers only)", "category": "",
             **{g: "" for g in GROUPS + ["Total"]}})
for cat in EXPERIENCE_ROWS:
    r = {"characteristic": "", "category": cat}
    for g, sub in by_group(df):
        base = sub[sub["group"] != "Patient"]
        k = base["experience"].isna().sum() if cat == "Not answered" else (base["experience"] == cat).sum()
        r[g] = n_pct(k, len(base))
    r["Patient"] = "-"
    rows.append(r)

table1 = pd.DataFrame(rows)
table1.to_csv(RESULTS / "table1_characteristics.csv", index=False)
table1

,characteristic,category,Patient,Healthcare professional,AI developer,Total
0,"Participants, n",,48,58,147,253
1,Gender,,,,,
2,,Male,25 (52.1),26 (44.8),95 (64.6),146 (57.7)
3,,Female,23 (47.9),32 (55.2),52 (35.4),107 (42.3)
4,,Non-binary,0 (0.0),0 (0.0),0 (0.0),0 (0.0)
5,,Prefer not to say,0 (0.0),0 (0.0),0 (0.0),0 (0.0)
6,Age group (years),,,,,
7,,18-24,11 (22.9),21 (36.2),78 (53.1),110 (43.5)
8,,25-34,16 (33.3),19 (32.8),41 (27.9),76 (30.0)
9,,35-44,18 (37.5),14 (24.1),25 (17.0),57 (22.5)


## Every answer to every question, by group

In [5]:
OPINION_ITEMS = ["bias_concern", "privacy_protection", "explanation_importance", "inform_patients", "trust",
                 "autonomy_undermined", "guidelines_confidence", "no_human_life_death", "disparities_concern"]

rows = []
for var in OPINION_ITEMS + ["familiarity"]:
    for cat in CATEGORIES[var]:
        r = {"variable": var, "answer": cat}
        for g, sub in by_group(df):
            k = int((sub[var] == cat).sum())
            r[g + "_n"] = k
            r[g + "_pct"] = round(100 * k / len(sub), 1)
        rows.append(r)
distributions = pd.DataFrame(rows)

# Each question's answers must add up to the group size.
for var in OPINION_ITEMS:
    t = distributions[distributions["variable"] == var]
    for g, sub in by_group(df):
        assert t[g + "_n"].sum() == len(sub), var + " " + g

distributions.to_csv(RESULTS / "item_distributions.csv", index=False)
distributions.head(15)

,variable,answer,Patient_n,Patient_pct,Healthcare professional_n,Healthcare professional_pct,AI developer_n,AI developer_pct,Total_n,Total_pct
0,bias_concern,Not concerned,8,16.7,8,13.8,14,9.5,30,11.9
1,bias_concern,Neutral,14,29.2,21,36.2,54,36.7,89,35.2
2,bias_concern,Somewhat concerned,12,25.0,12,20.7,35,23.8,59,23.3
3,bias_concern,Concerned,9,18.8,14,24.1,34,23.1,57,22.5
4,bias_concern,Extremely concerned,5,10.4,3,5.2,10,6.8,18,7.1
5,privacy_protection,Strongly disagree,3,6.2,1,1.7,9,6.1,13,5.1
6,privacy_protection,Disagree,8,16.7,8,13.8,22,15.0,38,15.0
7,privacy_protection,Neutral,19,39.6,24,41.4,58,39.5,101,39.9
8,privacy_protection,Agree,15,31.2,21,36.2,46,31.3,82,32.4
9,privacy_protection,Strongly agree,3,6.2,4,6.9,12,8.2,19,7.5


## Key percentages with 95% confidence intervals

The main percentage for each question. For Q7, Q9, Q14 and Q16 the "somewhat or higher" share is the main description and the "top two" share is reported as well.

In [6]:
KEY = [  # (label, rule, type)
    ("Q7 bias: any concern (Somewhat or more)", lambda d: d["bias_concern_any"] == 1, "main"),
    ("Q7 bias: strong concern (top two)", lambda d: d["bias_concern_strong"] == 1, "sensitivity"),
    ("Q8 privacy is protected: agree or strongly agree", lambda d: d["privacy_protection"].isin(["Agree", "Strongly agree"]), "main"),
    ("Q8 privacy is protected: disagree or strongly disagree", lambda d: d["privacy_protection"].isin(["Disagree", "Strongly disagree"]), "main"),
    ("Q9 explanations: any importance (Somewhat or more)", lambda d: d["explanation_importance_any"] == 1, "main"),
    ("Q9 explanations: strong importance (top two)", lambda d: d["explanation_importance_strong"] == 1, "sensitivity"),
    ("Q10 patients should always be informed: Yes", lambda d: d["inform_patients"] == "Yes", "main"),
    ("Q11 accountable: AI developers", lambda d: d["acc_ai_developers"] == 1, "main"),
    ("Q11 accountable: healthcare professional", lambda d: d["acc_healthcare_professional"] == 1, "main"),
    ("Q11 accountable: hospital or clinic", lambda d: d["acc_hospital_clinic"] == 1, "main"),
    ("Q12 trust: no trust", lambda d: d["trust"] == "No trust", "main"),
    ("Q12 trust: complete trust", lambda d: d["trust"] == "Complete trust", "main"),
    ("Q13 autonomy could be undermined: agree or strongly agree", lambda d: d["autonomy_undermined"].isin(["Agree", "Strongly agree"]), "main"),
    ("Q14 guidelines are enough: any confidence (Somewhat or more)", lambda d: d["guidelines_confidence_any"] == 1, "main"),
    ("Q14 guidelines are enough: strong confidence (top two)", lambda d: d["guidelines_confidence_strong"] == 1, "sensitivity"),
    ("Q15 comfortable with no human: No", lambda d: d["no_human_life_death"] == "No", "main"),
    ("Q15 comfortable with no human: Yes", lambda d: d["no_human_life_death"] == "Yes", "main"),
    ("Q16 disparities: any concern (Somewhat or more)", lambda d: d["disparities_concern_any"] == 1, "main"),
    ("Q16 disparities: strong concern (top two)", lambda d: d["disparities_concern_strong"] == 1, "sensitivity"),
]

rows = []
for label, rule, kind in KEY:
    for g, sub in by_group(df):
        k, n = int(rule(sub).sum()), len(sub)
        lo, hi = wilson(k, n)
        rows.append({"measure": label, "type": kind, "group": g, "n_yes": k, "n": n,
                     "pct": round(100 * k / n, 1), "ci_low": round(lo, 1), "ci_high": round(hi, 1)})
key = pd.DataFrame(rows)
key.to_csv(RESULTS / "key_percentages.csv", index=False)
key[key["group"] == "Total"]

,measure,type,group,n_yes,n,pct,ci_low,ci_high
3,Q7 bias: any concern (Somewhat or more),main,Total,134,253,53.0,46.8,59.0
7,Q7 bias: strong concern (top two),sensitivity,Total,75,253,29.6,24.4,35.5
11,Q8 privacy is protected: agree or strongly agree,main,Total,101,253,39.9,34.1,46.1
15,Q8 privacy is protected: disagree or strongly ...,main,Total,51,253,20.2,15.7,25.5
19,Q9 explanations: any importance (Somewhat or m...,main,Total,195,253,77.1,71.5,81.8
23,Q9 explanations: strong importance (top two),sensitivity,Total,166,253,65.6,59.6,71.2
27,Q10 patients should always be informed: Yes,main,Total,189,253,74.7,69.0,79.7
31,Q11 accountable: AI developers,main,Total,161,253,63.6,57.5,69.3
35,Q11 accountable: healthcare professional,main,Total,172,253,68.0,62.0,73.4
39,Q11 accountable: hospital or clinic,main,Total,138,253,54.5,48.4,60.6


## Ordered questions: median and interquartile range

Extra description only. Codes: privacy and autonomy 1 = strongly disagree to 5 = strongly agree; trust 1 = no trust to 3 = complete trust; familiarity 1 = not familiar at all to 5 = very familiar.

In [7]:
rows = []
for var in ["privacy_protection_code", "autonomy_undermined_code", "trust_code", "familiarity_code"]:
    for g, sub in by_group(df):
        s = sub[var].dropna()
        rows.append({"variable": var, "group": g, "n": len(s), "median": s.median(),
                     "q1": s.quantile(0.25), "q3": s.quantile(0.75)})
ordered_summary = pd.DataFrame(rows)
ordered_summary.to_csv(RESULTS / "ordered_item_summaries.csv", index=False)
ordered_summary.pivot(index="variable", columns="group", values="median")[GROUPS + ["Total"]]

group,Patient,Healthcare professional,AI developer,Total
variable,,,,
autonomy_undermined_code,3.0,3.0,3.0,3.0
familiarity_code,2.0,3.0,3.0,3.0
privacy_protection_code,3.0,3.0,3.0,3.0
trust_code,2.0,2.0,2.0,2.0


## How participants combined their answers on accountability (Q11)

In [8]:
PARTIES = {"acc_ai_developers": "AI developers", "acc_healthcare_professional": "Healthcare professional",
           "acc_hospital_clinic": "Hospital or clinic"}
df["acc_combination"] = df.apply(lambda row: " + ".join(PARTIES[c] for c in PARTIES if row[c] == 1), axis=1)

rows = []
for kind, col in [("number chosen", "acc_n_selected"), ("combination", "acc_combination")]:
    for value in sorted(df[col].unique(), key=str):
        rows.append({"pattern_type": kind, "pattern": str(value),
                     **{g: n_pct((sub[col] == value).sum(), len(sub)) for g, sub in by_group(df)}})
patterns = pd.DataFrame(rows)
patterns.to_csv(RESULTS / "accountability_patterns.csv", index=False)
patterns

,pattern_type,pattern,Patient,Healthcare professional,AI developer,Total
0,number chosen,1,21 (43.8),33 (56.9),57 (38.8),111 (43.9)
1,number chosen,2,10 (20.8),14 (24.1),42 (28.6),66 (26.1)
2,number chosen,3,17 (35.4),11 (19.0),48 (32.7),76 (30.0)
3,combination,AI developers,7 (14.6),8 (13.8),32 (21.8),47 (18.6)
4,combination,AI developers + Healthcare professional,2 (4.2),6 (10.3),19 (12.9),27 (10.7)
5,combination,AI developers + Healthcare professional + Hosp...,17 (35.4),11 (19.0),48 (32.7),76 (30.0)
6,combination,AI developers + Hospital or clinic,2 (4.2),4 (6.9),5 (3.4),11 (4.3)
7,combination,Healthcare professional,7 (14.6),16 (27.6),18 (12.2),41 (16.2)
8,combination,Healthcare professional + Hospital or clinic,6 (12.5),4 (6.9),18 (12.2),28 (11.1)
9,combination,Hospital or clinic,7 (14.6),9 (15.5),7 (4.8),23 (9.1)
